# Lab 5　一家人上場：戒菸會讓人胖多少？AIPW、TMLE、DML、CV-TMLE

Lab 2–4 在模擬資料上看懂了每一塊誤差。這裡換成真實資料，復刻 Karim (2026) Three Routes to One Answer 的 NHEFS 主線：同一組 nuisance，接成 AIPW、TMLE；加上 cross-fitting，就是 DML 和 CV-TMLE。沒有真值，只能靠診斷（overlap、balance、敏感度）判斷答案可不可信。對應投影片 80–88。

> **怎麼跑：** 這本筆記本用 **R** 執行（Colab 會自動選 R runtime；如果右上角不是 R，到「執行階段 → 變更執行階段類型」選 R）。從上往下依序執行：第一格下載資料、第二格載入工具箱，之後每一格都可以改了再跑。想一次跑完：「執行階段 → 全部執行」。
>
> 網頁版（不用登入、直接在瀏覽器跑）：[https://kiwi710680.github.io/Pokemon-CP/05_nhefs.html](https://kiwi710680.github.io/Pokemon-CP/05_nhefs.html)

In [ ]:
# Setup: download the data used by this lab (from https://raw.githubusercontent.com/kiwi710680/Pokemon-CP/main/) and set plot size
BASE <- "https://raw.githubusercontent.com/kiwi710680/Pokemon-CP/main/"
files <- c("data/nhefs.csv")
for (f in files) if (!file.exists(f)) {
  dir.create(dirname(f), recursive = TRUE, showWarnings = FALSE)
  for (try in 1:3) {                                   # retry: a single network hiccup should not stop the lab
    ok <- tryCatch(download.file(paste0(BASE, f), f, quiet = TRUE, mode = "wb") == 0, error = function(e) FALSE)
    if (ok) break else Sys.sleep(2)
  }
  if (!ok) stop("could not download ", f, " - check the internet connection and run this cell again")
}
options(repr.plot.width = 12, repr.plot.height = 7, repr.plot.res = 100, width = 110, digits = 4)
cat(if (length(files)) paste("ready:", length(files), "data file(s)") else "ready: this lab simulates its own data", "\n")

## 工具箱

這一頁用到的 R 函式都在下面這一格：估計器、資料讀取、畫圖小工具。先執行它；想知道某個函式做什麼，回來這裡找。

In [ ]:
# ---- estimators: one set of nuisance predictions -> different answers ----
# Notation (same as the slides): O = (X, A, Y)
#   Q(a, X) = E[Y | A = a, X]            outcome regression   -> nu$Q1, nu$Q0
#   g(X)    = P(A = 1 | X)               propensity score     -> nu$g
#   H(A, X) = A / g - (1 - A) / (1 - g)  clever covariate
#   D(O; g, Q, psi) = H (Y - Q_A) + Q1 - Q0 - psi   efficient influence function of the ATE
expit <- function(x) 1 / (1 + exp(-x))
bound <- function(p, lo = 0.025, hi = 0.975) pmin(pmax(p, lo), hi)

est_plugin <- function(nu) mean(nu$Q1 - nu$Q0)               # g-computation: predict twice, subtract, average

est_ipw_ht <- function(dat, nu)                               # Horvitz-Thompson IPTW: divide by n
  mean(dat$A * dat$Y / nu$g - (1 - dat$A) * dat$Y / (1 - nu$g))

est_ipw <- function(dat, nu) {                                # Hajek (stabilized) IPTW: divide by the sum of weights
  w1 <- dat$A / nu$g; w0 <- (1 - dat$A) / (1 - nu$g)
  sum(w1 * dat$Y) / sum(w1) - sum(w0 * dat$Y) / sum(w0)
}

eif <- function(dat, nu, psi) {
  H  <- dat$A / nu$g - (1 - dat$A) / (1 - nu$g)
  QA <- ifelse(dat$A == 1, nu$Q1, nu$Q0)
  H * (dat$Y - QA) + nu$Q1 - nu$Q0 - psi
}

wald <- function(psi, D) {                                    # SE from the EIF: sqrt(mean(D^2) / n)
  se <- sqrt(mean(D^2) / length(D))
  c(est = psi, se = se, lo = psi - 1.96 * se, hi = psi + 1.96 * se)
}

est_aipw <- function(dat, nu) {                               # one-step: psi_plug + P_n D  (solves P_n D = 0)
  psi <- mean(eif(dat, nu, 0))
  wald(psi, eif(dat, nu, psi))
}

est_tmle <- function(dat, nu, ybounds = range(dat$Y)) {       # targeting: move Q-hat along H until P_n D = 0
  lo <- ybounds[1]; hi <- ybounds[2]
  sc <- function(y) pmin(pmax((y - lo) / (hi - lo), 1e-6), 1 - 1e-6)
  Ys <- (dat$Y - lo) / (hi - lo)
  Q1s <- sc(nu$Q1); Q0s <- sc(nu$Q0); QAs <- ifelse(dat$A == 1, Q1s, Q0s)
  H  <- dat$A / nu$g - (1 - dat$A) / (1 - nu$g)
  fl <- suppressWarnings(glm(Ys ~ -1 + H, offset = qlogis(QAs), family = quasibinomial()))
  eps <- unname(coef(fl))
  Q1t <- expit(qlogis(Q1s) + eps / nu$g)
  Q0t <- expit(qlogis(Q0s) - eps / (1 - nu$g))
  nu_t <- data.frame(Q1 = lo + (hi - lo) * Q1t, Q0 = lo + (hi - lo) * Q0t, g = nu$g)
  psi <- mean(nu_t$Q1 - nu_t$Q0)
  c(wald(psi, eif(dat, nu_t, psi)), eps = eps)
}

weight_summary <- function(dat, nu) {                         # largest weight and effective sample size (ESS)
  w <- ifelse(dat$A == 1, 1 / nu$g, 1 / (1 - nu$g))
  c(max_weight = max(w),
    ESS_treated = sum(w[dat$A == 1])^2 / sum(w[dat$A == 1]^2), n_treated = sum(dat$A),
    ESS_control = sum(w[dat$A == 0])^2 / sum(w[dat$A == 0]^2), n_control = sum(1 - dat$A))
}

# ---- NHEFS: data, candidate learners, sl_lite, nuisances ----
load_nhefs <- function(path = "data/nhefs.csv") {
  d <- subset(read.csv(path), !is.na(wt82_71))                # complete cases for the outcome
  data.frame(Y = d$wt82_71, A = d$qsmk, sex = factor(d$sex), race = factor(d$race), age = d$age,
             education = factor(d$education), smokeintensity = d$smokeintensity, smokeyrs = d$smokeyrs,
             exercise = factor(d$exercise), active = factor(d$active), wt71 = d$wt71)
}
rhs <- list(                                                  # each learner is just a formula
  mean   = "1",
  main   = "as.numeric(sex) + as.numeric(race) + age + as.numeric(education) + smokeintensity + smokeyrs + as.numeric(exercise) + as.numeric(active) + wt71",
  hr     = "sex + race + age + I(age^2) + education + smokeintensity + I(smokeintensity^2) + smokeyrs + I(smokeyrs^2) + exercise + active + wt71 + I(wt71^2)",
  spline = "sex + race + education + exercise + active + splines::ns(age, 3) + splines::ns(smokeintensity, 3) + splines::ns(smokeyrs, 3) + splines::ns(wt71, 3)")
q_formula <- function(r) as.formula(if (r == "1") "Y ~ A" else paste("Y ~ A + A:smokeintensity +", r))
g_formula <- function(r) as.formula(paste("A ~", r))
fit_pred <- function(train, newdata, learner, what) {
  if (what == "g") predict(glm(g_formula(rhs[[learner]]), family = binomial, data = train), newdata, type = "response")
  else predict(lm(q_formula(rhs[[learner]]), data = train), newdata)
}
convex_weights <- function(Z, y) {
  obj <- function(b) { w <- exp(c(0, b)); w <- w / sum(w); mean((y - Z %*% w)^2) }
  b <- optim(rep(0, ncol(Z) - 1), obj, method = "BFGS")$par
  w <- exp(c(0, b)); setNames(w / sum(w), colnames(Z))
}
sl_lite <- function(train, newdata, what, learners = names(rhs), V = 5) {   # inner V-fold CV -> convex weights
  y <- if (what == "g") train$A else train$Y
  fold <- sample(rep(seq_len(V), length.out = nrow(train)))
  Z <- matrix(NA_real_, nrow(train), length(learners), dimnames = list(NULL, learners))
  for (v in seq_len(V)) for (l in learners) Z[fold == v, l] <- fit_pred(train[fold != v, ], train[fold == v, ], l, what)
  w <- convex_weights(Z, y)
  structure(as.vector(sapply(learners, function(l) fit_pred(train, newdata, l, what)) %*% w), weights = w)
}
nhefs_nuisance <- function(train, newdata, learner = "sl_lite", bounds = c(0.025, 0.975)) {
  pr <- function(nd, what) if (learner == "sl_lite") sl_lite(train, nd, what) else fit_pred(train, nd, learner, what)
  g_raw <- pr(newdata, "g")
  data.frame(Q1 = pr(transform(newdata, A = 1), "Q"), Q0 = pr(transform(newdata, A = 0), "Q"),
             g = bound(g_raw, bounds[1], bounds[2]), g_raw = g_raw)
}
nhefs_crossfit <- function(dat, learner = "sl_lite", K = 5, seed = 1, bounds = c(0.025, 0.975)) {
  set.seed(seed)
  fold <- sample(rep(seq_len(K), length.out = nrow(dat)))
  nu <- data.frame(Q1 = rep(NA_real_, nrow(dat)), Q0 = NA_real_, g = NA_real_, g_raw = NA_real_)
  for (k in seq_len(K)) nu[fold == k, ] <- nhefs_nuisance(dat[fold != k, ], dat[fold == k, ], learner, bounds)
  nu
}
nhefs_panel <- function(dat, nu, label) rbind(
  data.frame(estimator = "g-computation", scheme = label, est = est_plugin(nu), se = NA, lo = NA, hi = NA),
  data.frame(estimator = "IPTW (Hajek)",  scheme = label, est = est_ipw(dat, nu), se = NA, lo = NA, hi = NA),
  data.frame(estimator = "AIPW",          scheme = label, t(est_aipw(dat, nu))),
  data.frame(estimator = "TMLE",          scheme = label, t(est_tmle(dat, nu)[1:4])))
nhefs_X <- function(dat) {                                    # numeric covariates (dummies) for balance checks
  X <- as.data.frame(model.matrix(~ sex + race + age + education + smokeintensity + smokeyrs + exercise + active + wt71 - 1, data = dat))
  X[, setdiff(names(X), "sex0"), drop = FALSE]
}

# ---- real data: loaders return list(name, X, A, Y, binary) ----
load_rhc <- function(path = "data/rhc.csv") {
  d <- read.csv(path, stringsAsFactors = TRUE)
  d[[1]] <- NULL                                              # first column is a row number
  A <- as.integer(d$swang1 == "RHC"); Y <- as.integer(d$dth30 == "Yes")
  drop <- c("swang1", "death", "dth30", "sadmdte", "dschdte", "dthdte", "lstctdte",   # same drop list as Karim (2026)
            "t3d30", "ptid", "cat2", "adld3p", "urin1")
  covs <- droplevels(d[, setdiff(names(d), drop), drop = FALSE])
  X <- as.data.frame(model.matrix(~ . - 1, data = covs))
  colnames(X) <- make.names(colnames(X), unique = TRUE)
  list(name = "RHC", X = X, A = A, Y = Y, binary = TRUE)
}
load_ihdp <- function(r = 1, dir = "data/ihdp") {
  d <- read.csv(file.path(dir, sprintf("ihdp_npci_%d.csv", r)), header = FALSE)
  names(d) <- c("treatment", "y_factual", "y_cfactual", "mu0", "mu1", paste0("x", 1:25))
  list(name = sprintf("IHDP #%d", r), X = d[, paste0("x", 1:25)], A = d$treatment, Y = d$y_factual,
       binary = FALSE, truth = mean(d$mu1 - d$mu0), mu0 = d$mu0, mu1 = d$mu1)
}
read_nsw <- function(file) {
  d <- read.table(file, header = FALSE)
  names(d) <- c("treat", "age", "educ", "black", "hisp", "married", "nodegree", "re74", "re75", "re78")
  d
}
load_lalonde <- function(setting = c("psid", "experimental"), dir = "data/lalonde") {
  setting <- match.arg(setting)
  tr <- read_nsw(file.path(dir, "nswre74_treated.txt")); ct <- read_nsw(file.path(dir, "nswre74_control.txt"))
  benchmark <- mean(tr$re78) - mean(ct$re78)                   # randomized: the simple difference is the answer
  d <- if (setting == "experimental") rbind(tr, ct) else rbind(tr, read_nsw(file.path(dir, "psid_controls.txt")))
  covs <- c("age", "educ", "black", "hisp", "married", "nodegree", "re74", "re75")
  list(name = paste0("LaLonde (", setting, ")"), X = d[, covs], A = d$treat, Y = d$re78, binary = FALSE, benchmark = benchmark)
}

# ---- learners: "glm" (main effects) or "rf" (random forest) ----
learn_g <- function(X, A, newX, learner = "glm", ntree = 100) {
  if (learner == "glm") {
    fit <- suppressWarnings(glm(A ~ ., data = data.frame(A = A, X), family = binomial))
    return(unname(suppressWarnings(predict(fit, newX, type = "response"))))
  }
  fit <- randomForest::randomForest(x = X, y = factor(A, levels = 0:1), ntree = ntree)
  unname(predict(fit, newX, type = "prob")[, "1"])
}
learn_Q <- function(X, A, Y, newX, binary, learner = "glm", ntree = 100) {
  tr <- data.frame(A = A, X); n1 <- data.frame(A = 1, newX); n0 <- data.frame(A = 0, newX)
  if (learner == "glm") {
    fit <- if (binary) suppressWarnings(glm(Y ~ ., data = data.frame(Y = Y, tr), family = binomial))
           else lm(Y ~ ., data = data.frame(Y = Y, tr))
    pr <- function(nd) unname(suppressWarnings(predict(fit, nd, type = "response")))
    return(list(Q1 = pr(n1), Q0 = pr(n0)))
  }
  if (binary) {
    fit <- randomForest::randomForest(x = tr, y = factor(Y, levels = 0:1), ntree = ntree)
    pr <- function(nd) unname(predict(fit, nd, type = "prob")[, "1"])
  } else {
    fit <- randomForest::randomForest(x = tr, y = Y, ntree = ntree)
    pr <- function(nd) unname(predict(fit, nd))
  }
  list(Q1 = pr(n1), Q0 = pr(n0))
}

# full: train on everyone, predict the same people.  crossfit: each prediction comes from the other K - 1 folds
nuisance <- function(d, learner = "glm", scheme = c("full", "crossfit"), K = 5, seed = 1,
                     bounds = c(0.025, 0.975), ntree = 100) {
  scheme <- match.arg(scheme)
  n <- length(d$A)
  set.seed(seed)
  fold <- if (scheme == "full") rep(1L, n) else sample(rep(seq_len(K), length.out = n))
  out <- data.frame(Q1 = numeric(n), Q0 = numeric(n), g_raw = numeric(n))
  for (k in unique(fold)) {
    te <- fold == k; tr <- if (scheme == "full") rep(TRUE, n) else !te
    q <- learn_Q(d$X[tr, , drop = FALSE], d$A[tr], d$Y[tr], d$X[te, , drop = FALSE], d$binary, learner, ntree)
    out$Q1[te] <- q$Q1; out$Q0[te] <- q$Q0
    out$g_raw[te] <- learn_g(d$X[tr, , drop = FALSE], d$A[tr], d$X[te, , drop = FALSE], learner, ntree)
  }
  if (d$binary) { out$Q1 <- pmin(pmax(out$Q1, 0), 1); out$Q0 <- pmin(pmax(out$Q0, 0), 1) }
  out$g <- pmin(pmax(out$g_raw, bounds[1]), bounds[2])
  out$fold <- fold
  out
}

panel <- function(d, nu, label = "") {                        # one set of nuisances -> four estimators
  dd <- list(A = d$A, Y = d$Y)
  a <- est_aipw(dd, nu); t <- est_tmle(dd, nu, ybounds = if (d$binary) c(0, 1) else range(d$Y))
  data.frame(scheme = label, estimator = c("g-computation", "IPTW (Hajek)", "AIPW", "TMLE"),
             est = c(est_plugin(nu), est_ipw(dd, nu), a[["est"]], t[["est"]]),
             se = c(NA, NA, a[["se"]], t[["se"]]), lo = c(NA, NA, a[["lo"]], t[["lo"]]), hi = c(NA, NA, a[["hi"]], t[["hi"]]))
}
residual_check <- function(d, nu) {                           # is the learner grading its own exam?
  QA <- ifelse(d$A == 1, nu$Q1, nu$Q0); H <- d$A / nu$g - (1 - d$A) / (1 - nu$g)
  c(resid_rms = sqrt(mean((d$Y - QA)^2)), correction = mean(H * (d$Y - QA)))
}

# ---- diagnostics: overlap and balance ----
smd <- function(x, A, w = rep(1, length(A))) {                # standardized mean difference
  m1 <- weighted.mean(x[A == 1], w[A == 1]); m0 <- weighted.mean(x[A == 0], w[A == 0])
  s <- sqrt((var(x[A == 1]) + var(x[A == 0])) / 2)
  if (s == 0) 0 else (m1 - m0) / s
}
balance_table <- function(X, A, g) {
  w <- ifelse(A == 1, 1 / g, 1 / (1 - g))
  data.frame(variable = names(X), before = sapply(X, smd, A = A), after_IPW = sapply(X, smd, A = A, w = w), row.names = NULL)
}
overlap_summary <- function(g_raw, A, bounds = c(0.025, 0.975)) {
  gb <- pmin(pmax(g_raw, bounds[1]), bounds[2]); w <- ifelse(A == 1, 1 / gb, 1 / (1 - gb))
  c(min_g = min(g_raw), max_g = max(g_raw), pct_clipped = 100 * mean(g_raw < bounds[1] | g_raw > bounds[2]),
    max_weight = max(w), ESS_treated = sum(w[A == 1])^2 / sum(w[A == 1]^2), ESS_control = sum(w[A == 0])^2 / sum(w[A == 0]^2))
}

# ---- plotting helpers (base graphics only) ----
COL <- list(red = "#D9480F", teal = "#0B7285", ink = "#1C1B19", muted = "#6B665E",
            gold = "#C98A0C", plum = "#7B3F8C", rule = "#DDD6CA")
lab_par <- function(...) par(mar = c(4.5, 5, 1.5, 1), las = 1, bty = "n", col.axis = COL$muted, ...)

overlap_plot <- function(g_raw, A, xlab = "estimated propensity score  g-hat(X)", labels = c("A = 1", "A = 0"), breaks = 40) {
  br <- seq(0, 1, length.out = breaks + 1)                    # mirrored histogram: treated up, control down
  h1 <- hist(pmin(pmax(g_raw[A == 1], 0), 1), breaks = br, plot = FALSE)$counts
  h0 <- hist(pmin(pmax(g_raw[A == 0], 0), 1), breaks = br, plot = FALSE)$counts
  lab_par()
  plot(NA, xlim = c(0, 1), ylim = c(-max(h0), max(h1)), axes = FALSE, xlab = xlab, ylab = "count")
  axis(1); yt <- pretty(c(-max(h0), max(h1))); axis(2, at = yt, labels = abs(yt))
  rect(br[-length(br)], 0, br[-1], h1, col = adjustcolor(COL$red, .8), border = "white")
  rect(br[-length(br)], 0, br[-1], -h0, col = adjustcolor(COL$teal, .8), border = "white")
  abline(h = 0)
  text(0.98, max(h1) * .85, labels[1], adj = 1, col = COL$red, font = 2)
  text(0.98, -max(h0) * .85, labels[2], adj = 1, col = COL$teal, font = 2)
}

forest_plot <- function(lab, est, lo = NA, hi = NA, ref = NULL, ref_lab = "", col = COL$ink, xlab = "estimate", xlim = NULL) {
  n <- length(est); y <- n:1
  rng <- range(c(est, lo, hi, ref), na.rm = TRUE)
  if (is.null(xlim)) xlim <- rng + c(-1, 1) * diff(rng) * 0.08
  op <- par(mar = c(4.5, min(22, max(8, 0.62 * max(nchar(lab)) + 1)), 1.5, 1), las = 1, bty = "n", col.axis = COL$muted)
  on.exit(par(op))
  plot(NA, xlim = xlim, ylim = c(0.5, n + 0.5), axes = FALSE, xlab = xlab, ylab = "")
  abline(h = y, col = "#F0EBE3")
  if (!is.null(ref)) {
    abline(v = ref, lty = 2, col = COL$muted); pos <- (ref[1] - xlim[1]) / diff(xlim)
    mtext(ref_lab, 3, at = ref[1], adj = if (pos > 0.7) 1 else if (pos < 0.3) 0 else 0.5, cex = .8, col = COL$muted)
  }
  ok <- !is.na(lo)
  segments(lo[ok], y[ok], hi[ok], y[ok], lwd = 3, col = rep_len(col, n)[ok])
  points(est, y, pch = 19, cex = 1.5, col = rep_len(col, n))
  axis(1); axis(2, at = y, labels = lab, tick = FALSE, col.axis = COL$ink)
}

| 格 | Karim (2026) 的步驟 | 投影片 |
|---|---|---|
| 01–02 | Step 0：固定題目；看兩組差在哪 | 37–40 |
| 03–04 | Step 1：先診斷 overlap 與 balance，再選 estimator | 40、62 |
| 05–07 | Step 2–4：共用 nuisance，full sample vs cross-fit | 80–86 |
| 08–10 | Step 6：敏感度（截斷、learner、切分） | 77–79、87–88 |

> **一家人怎麼分？**兩個問題就分完了：(1) 怎麼把 EIF 變成一個數字——加回去（AIPW 家族）或推 Q̂（TMLE 家族）；(2) nuisance 怎麼訓練——full sample、cross-fit、double cross-fit。AIPW + cross-fit = DML，TMLE + cross-fit = CV-TMLE（投影片 81 那張 2×3 表）。
> 
> 與原論文的差異：原文的 SuperLearner 用 5 個 learner（mean、GLM、glmnet、ranger、earth）；這裡用 base R 寫的 **sl_lite**：mean、主效應 GLM、Hernán–Robins GLM（加平方項）、樣條 GLM 的凸組合，權重由內層 5 折 CV 決定。數字不會和論文逐位相同，但應落在同一範圍（論文 3.32–3.42 kg）。

## 01　資料與問題　<small>（投影片 37–40）</small>

A = 1971–1982 間戒菸（qsmk），Y = 同期間體重變化（kg），X = 性別、種族、年齡、教育、吸菸強度、菸齡、運動、活動量、1971 年體重。只用 Y 沒缺的 1,566 人（complete case），所以 estimand 是「這 1,566 人」的 ATE，不是全美國人。

In [ ]:
dat <- load_nhefs("data/nhefs.csv")
c(n = nrow(dat), quit = sum(dat$A), unadjusted_diff_kg = mean(dat$Y[dat$A == 1]) - mean(dat$Y[dat$A == 0]))

## 02　戒菸的人跟沒戒菸的人，本來就不一樣

標準化平均差（SMD）＝兩組平均差除以合併標準差。|SMD| &gt; 0.1 通常就算「兩組不像」。

In [ ]:
X <- nhefs_X(dat)
grp <- data.frame(mean_quit = sapply(X[dat$A == 1, ], mean), mean_no_quit = sapply(X[dat$A == 0, ], mean),
                  SMD = sapply(X, smd, A = dat$A))
round(grp[order(-abs(grp$SMD)), ], 2)

> **看什麼：**
>
> 戒菸的人平均年紀比較大（SMD 0.28）、每天抽的菸比較少、白人比例比較高。年紀和體重變化本身有關——不調整就比，2.5 kg 這個數字混進了「年紀」的效果。

## 03　先診斷 overlap，再選 estimator　<small>（投影片 40、62）</small>

In [ ]:
set.seed(1)
nf <- nhefs_nuisance(dat, dat, "hr")        # start with the Hernan-Robins GLM: fastest
round(overlap_summary(nf$g_raw, dat$A), 3)
overlap_plot(nf$g_raw, dat$A, xlab = "g-hat(X) = P(quit | X)", labels = c("quit (403)", "did not quit (1163)"))

> **看什麼：**
>
> ĝ 從 0.05 到 0.78，兩組廣泛重疊；截斷在 [0.025, 0.975] 一個人都沒有被截到（論文：0%）。最大權重 17、戒菸組有效樣本數約 326（原本 403）——權重健康。這是 Karim 選 NHEFS 當「乾淨教學案例」的原因。

## 04　加權之後，兩組像了嗎？（balance）

用 1/ĝ（戒菸組）、1/(1−ĝ)（未戒菸組）加權，再算一次 SMD。紅點是加權前，藍點是加權後——這張圖叫 love plot。

In [ ]:
bt <- balance_table(X, dat$A, nf$g)
bt <- bt[order(abs(bt$before)), ]
lab_par(mar = c(4.5, 9, 1, 1))
plot(NA, xlim = c(0, 0.32), ylim = c(1, nrow(bt)), axes = FALSE, xlab = "|SMD|", ylab = "")
abline(v = 0.1, lty = 2, col = "grey60")
segments(abs(bt$after_IPW), seq_len(nrow(bt)), abs(bt$before), seq_len(nrow(bt)), col = "grey80")
points(abs(bt$before), seq_len(nrow(bt)), pch = 19, col = COL$red)
points(abs(bt$after_IPW), seq_len(nrow(bt)), pch = 19, col = COL$teal)
axis(1); axis(2, at = seq_len(nrow(bt)), labels = bt$variable, tick = FALSE, col.axis = COL$ink)
legend("bottomright", c("before weighting", "after IPW"), pch = 19, col = c(COL$red, COL$teal), bty = "n")

> **看什麼：**
>
> 紅點有 8 個超過 0.1（年紀最大，0.28），加權後藍點全部縮到 0.04 以下。balance 是 ĝ「有沒有把兩組變像」的檢查——它只看得到量到的變數，看不到沒量到的混雜（exchangeability 還是假設）。

## 05　Full sample：同一組 nuisance，四種接法　<small>（投影片 83–85）</small>

In [ ]:
p_full <- nhefs_panel(dat, nf, "full sample")
p_full[, -2]
plug <- est_plugin(nf)
c(plugin = plug, AIPW_correction = est_aipw(dat, nf)[["est"]] - plug, TMLE_epsilon = est_tmle(dat, nf)[["eps"]])

> **看什麼：**
>
> AIPW 和 TMLE 幾乎一樣（差在小數點第三位）：AIPW 直接把修正項加上去，TMLE 先把 Q̂ 往 H 的方向推 ε 再 plug in。Q̂ 本來就不差（Hernán–Robins 的模型），所以修正項和 ε 都很小——和 Lab 3 的糖果是同一個故事。

## 06　換成 sl_lite，並做 cross-fitting（= DML / CV-TMLE）　<small>（投影片 78、86）</small>

每個人的 Q̂、ĝ 都由沒看過他的四折資料訓練；SuperLearner 的內層 CV 也放在外層的訓練折裡面。瀏覽器裡約 1 分鐘。

In [ ]:
nc <- nhefs_crossfit(dat, learner = "sl_lite", K = 5, seed = 1)
p_cf <- nhefs_panel(dat, nc, "cross-fit K=5")
p_cf[, -2]
set.seed(1); w <- attr(sl_lite(dat, dat[1:2, ], "Q"), "weights"); round(w, 3)   # sl_lite weights for Q

> **看什麼：**
>
> DML 和 CV-TMLE 約 3.4 kg，CI 約 2.5–4.3。sl_lite 把 7 成權重給 Hernán–Robins 模型（hr）——資料自己挑出了教科書的模型。瀏覽器和本機 R 可能差 0.04 左右：sl_lite 內層 CV 的隨機切法不完全一樣，第 10 格會看到這種「換切法」的晃動本來就有 0.1 左右。

## 07　畫在一起，和論文對照

In [ ]:
ours <- rbind(transform(p_full, src = "ours full"), transform(p_cf, src = "ours cross-fit"))
karim <- data.frame(estimator = c("g-computation", "IPTW (Hajek)", "AIPW", "TMLE", "AIPW", "TMLE"),
                    est = c(3.27, 3.19, 3.32, 3.33, 3.38, 3.39), lo = c(NA, NA, 2.50, 2.50, 2.41, 2.41),
                    hi = c(NA, NA, 4.14, 4.15, 4.36, 4.36),
                    src = c(rep("Karim full", 4), "Karim DML (r=100)", "Karim CV-TMLE (r=100)"))
lab <- c(paste(ours$src, ours$estimator), paste(karim$src, ifelse(grepl("r=100", karim$src), "", karim$estimator)))
forest_plot(lab, c(ours$est, karim$est), c(ours$lo, karim$lo), c(ours$hi, karim$hi),
            col = c(rep(COL$teal, 8), rep(COL$red, 6)), ref = 3.4, ref_lab = "Hernan-Robins ~ 3.4", xlab = "difference in weight change (kg)")

> **看什麼：**
>
> 藍色是我們、紅色是論文。所有雙重穩健的估計都在 3.3–3.5 kg 之間；它們之間的差距（約 0.1）遠小於 CI 的半寬（約 0.9）。**選哪條路，沒有選對題目和食材重要。**投影片 87–88 說 AIPW 和 TMLE 在有限樣本會不一樣——在 overlap 好、Q̂ 不差的資料上，差別小到看不出來。

## 08　敏感度一：propensity 截斷門檻

Karim 的練習題：把 [0.025, 0.975] 換成更鬆或更緊的門檻，看幾個人被截到、答案動了多少。

In [ ]:
tr <- t(sapply(list(c(.01, .99), c(.025, .975), c(.05, .95), c(.10, .90)), function(b) {
  nu <- nhefs_crossfit(dat, "hr", 5, 1, bounds = b); a <- est_aipw(dat, nu)
  c(lower = b[1], upper = b[2], pct_clipped = 100 * mean(nu$g_raw < b[1] | nu$g_raw > b[2]), AIPW = a[["est"]], SE = a[["se"]])
}))
round(tr, 3)

> **看什麼：**
>
> 門檻 0.025 以內沒有人被截到，0.05 也只截到 2 個人（0.13%），答案不動；收到 [0.10, 0.90] 才截到 5% 的人，答案也只動 0.02 kg。overlap 好的資料，截斷只是保險，不會改變結論——Lab 8 會看到相反的情況。

## 09　敏感度二：nuisance 用哪個 learner

In [ ]:
lr <- t(sapply(c("mean", "main", "hr", "spline"), function(l) { a <- est_aipw(dat, nhefs_crossfit(dat, l, 5, 1)); c(AIPW = a[["est"]], SE = a[["se"]]) }))
round(lr, 3)

> **看什麼：**
>
> 「mean」那列 Q̂ 和 ĝ 都只有截距（兩個模型都錯），AIPW 就退回未調整的 2.5 kg——和 Lab 4 第 03 格右下角那格一樣：雙重穩健不是兩個都錯也能救。主效應、Hernán–Robins、樣條三個都在 3.38–3.43：只要模型夠用，答案就穩。

## 10　敏感度三：換一個隨機切法，答案晃多少？　<small>（投影片 77–79）</small>

同一份資料、不同的 fold。Karim 用 100 次取中位數；這裡先跑 10 次（可以自己改大）。

In [ ]:
reps <- 10
r <- sapply(seq_len(reps), function(s) { nu <- nhefs_crossfit(dat, "hr", K = 5, seed = 100 + s)
  c(DML = est_aipw(dat, nu)[["est"]], CV_TMLE = est_tmle(dat, nu)[["est"]]) })
round(r, 3)
c(median_DML = median(r["DML", ]), sd_DML = sd(r["DML", ]))
lab_par(); hist(r["DML", ], breaks = 8, col = adjustcolor(COL$teal, .6), border = "white", main = "", xlab = "DML estimate (kg); one value per random split")
abline(v = median(r["DML", ]), lwd = 3, col = COL$red)

> **看什麼：**
>
> 10 種切法從 3.25 到 3.59，標準差約 0.1 kg（Karim：0.08），比抽樣 SE（約 0.5）小很多。但如果只比兩個任意的 seed，你可能是在比切法、不是在比方法——所以要預先寫好要重複幾次、怎麼彙整。

## 11　和論文對帳

| Estimator（Karim 2026, Fig. 4） | r | 估計值 (kg) | 95% CI |
|---|---|---|---|
| g-computation, full | – | 3.27 | 未報 |
| IPTW, full | – | 3.19 | 未報 |
| AIPW, full | – | 3.32 | 2.50–4.14 |
| TMLE, full | – | 3.33 | 2.50–4.15 |
| DML | 100 | 3.38 | 2.41–4.36 |
| CV-TMLE | 100 | 3.39 | 2.41–4.36 |

### NHEFS 沒有真值：幾個方法接近，代表答案對方法選擇不敏感，不代表識別假設成立。

那方法什麼時候會不一樣？Lab 6（RHC）看 learner 太會背答案、而且結果是二元的時候；Lab 7（IHDP）在知道真值的地方檢查 CI；Lab 8（LaLonde）看 overlap 不見的時候。